In [2]:
!pip install pandas scikit-learn mlxtend


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

df['Profitable'] = (df['Profit'] > 0).astype(int)

features = ['Category','Sub-Category','Region','Segment','Discount','Quantity']

X = pd.get_dummies(df[features])
y = df['Profitable']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = DecisionTreeClassifier(max_depth=5)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.91      0.79      0.84       384
           1       0.95      0.98      0.97      1615

    accuracy                           0.94      1999
   macro avg       0.93      0.88      0.91      1999
weighted avg       0.94      0.94      0.94      1999



In [6]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

customer = df.groupby('Customer ID').agg({
    'Sales':'sum',
    'Order ID':'count',
    'Discount':'mean',
    'Profit':'sum'
}).reset_index()

scaler = MinMaxScaler()
X = scaler.fit_transform(customer[['Sales','Order ID','Discount','Profit']])

kmeans = KMeans(n_clusters=4)
customer['Cluster'] = kmeans.fit_predict(X)

print(customer.head())

  Customer ID     Sales  Order ID  Discount    Profit  Cluster
0    AA-10315  5563.560        11  0.090909 -362.8825        1
1    AA-10375  1056.390        15  0.080000  277.3824        2
2    AA-10480  1790.512        12  0.016667  435.8274        1
3    AA-10645  5086.935        18  0.063889  857.8033        2
4    AB-10015   886.156         6  0.066667  129.3465        1


In [8]:
from mlxtend.frequent_patterns import apriori, association_rules

basket = df.groupby(['Order ID','Sub-Category'])['Quantity'].sum().unstack().fillna(0)

# Convert to boolean (fix warning too)
basket = basket > 0

# LOWER support (important fix)
frequent = apriori(basket, min_support=0.005, use_colnames=True)

# LOWER confidence
rules = association_rules(frequent, metric="confidence", min_threshold=0.2)

print(rules[['antecedents','consequents','support','confidence','lift']].head())

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

     antecedents consequents   support  confidence      lift
0  (Accessories)   (Binders)  0.032142    0.224234  0.853486
1  (Accessories)     (Paper)  0.030545    0.213092  0.896203
2   (Appliances)   (Binders)  0.025953    0.288248  1.097140
3   (Appliances)     (Paper)  0.021761    0.241685  1.016458
4          (Art)   (Binders)  0.029946    0.205198  0.781032


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag